# Lesson 12: Add token and position embeddings

A token ID tells the model *which symbol* appears, but attention alone does not tell it *where* that symbol appeared. GPT adds two learned vectors: one looked up by token ID and one looked up by position.

The vectors have the same width, so they can be added before entering the Transformer blocks. This lesson isolates that input representation before the next lesson connects it to the full decoder model.

For the array versions, start at [the C entry point](../c/src/embedding_main.c) and [the Java entry point](../java/src/main/java/learning/karpathy/gpt1/TokenAndPositionEmbeddingLesson.java). Both files show the table lookups and addition explicitly.

## Set up a tiny vocabulary and two sequences

We use five toy symbols, two sequences, four positions, and four channels. The embedding values follow simple deterministic formulas so the Python, C, and Java examples produce the same numbers. In a trained GPT, optimization learns these values from data.

In [ ]:
import torch
from torch import nn

torch.set_printoptions(precision=2, sci_mode=False)

BATCH_SIZE = 2
TIME_STEPS = 4
CHANNELS = 4
VOCABULARY = ["a", "b", "c", "d", "e"]
VOCAB_SIZE = len(VOCABULARY)

token_ids = torch.tensor(
    [
        [0, 1, 0, 2],
        [1, 2, 3, 0],
    ],
    dtype=torch.long,
)

print("Vocabulary:", {symbol: token_id for token_id, symbol in enumerate(VOCABULARY)})
print("Token IDs, shape [batch, time]:\n", token_ids)
print("Batch size:", BATCH_SIZE, "| time steps:", TIME_STEPS, "| model width:", CHANNELS)

## Step 1: Token IDs are lookup addresses

The IDs are categorical labels, not measurements. ID 4 is not “more” than ID 1. The vocabulary maps each symbol to a row in an embedding table.

In [ ]:
for batch, row in enumerate(token_ids.tolist()):
    symbols = [VOCABULARY[token_id] for token_id in row]
    print(f"batch {batch}: IDs {row} -> symbols {symbols}")

## Step 2: Look up a token vector

`nn.Embedding(V, C)` stores one trainable vector per vocabulary item. Indexing it with a `[B, T]` tensor returns `[B, T, C]`: each token ID is replaced by its row vector.

In [ ]:
token_embedding = nn.Embedding(VOCAB_SIZE, CHANNELS, dtype=torch.float64)

# Fixed demo values make the other languages easy to compare.
with torch.no_grad():
    for token_id in range(VOCAB_SIZE):
        for channel in range(CHANNELS):
            pattern = (token_id * 5 + channel * 3 + 1) % 11 - 5
            token_embedding.weight[token_id, channel] = pattern * 0.1

with torch.no_grad():
    token_vectors = token_embedding(token_ids)

print("Token embedding table shape:", tuple(token_embedding.weight.shape))
print("Token vectors shape:", tuple(token_vectors.shape))
print("Vectors for batch 0:\n", token_vectors[0])

## Step 3: Look up a position vector

The position table has one row for every allowed time step. `torch.arange(TIME_STEPS)` creates the position IDs `0, 1, 2, 3`. Looking them up gives `[T, C]`; this same position row is available to every sequence in the batch.

In [ ]:
position_embedding = nn.Embedding(TIME_STEPS, CHANNELS, dtype=torch.float64)

with torch.no_grad():
    for position in range(TIME_STEPS):
        for channel in range(CHANNELS):
            pattern = (position * 7 + channel * 2 + 2) % 9 - 4
            position_embedding.weight[position, channel] = pattern * 0.1

position_ids = torch.arange(TIME_STEPS)
with torch.no_grad():
    position_vectors = position_embedding(position_ids)

print("Position IDs:", position_ids.tolist())
print("Position embedding table shape:", tuple(position_vectors.shape))
print(position_vectors)

## Step 4: Add the two sources of information

The token vector is the same each time the same symbol appears. Adding its position vector lets the model distinguish those appearances. The result keeps shape `[B, T, C]`, which is the input shape expected by the Transformer blocks.

In [ ]:
with torch.no_grad():
    model_input = token_vectors + position_vectors

first_a = token_vectors[0, 0]
second_a = token_vectors[0, 2]
first_a_with_position = model_input[0, 0]
second_a_with_position = model_input[0, 2]

same_token_difference = (first_a - second_a).abs().max()
positioned_difference = (first_a_with_position - second_a_with_position).abs().max()

print("Combined model input shape:", tuple(model_input.shape))
print("Sequence 0, combined vectors:\n", model_input[0])
print(f"Same symbol at positions 0 and 2: token-vector difference = {same_token_difference.item():.6f}")
print(f"Same symbol at positions 0 and 2: combined-vector difference = {positioned_difference.item():.6f}")

torch.testing.assert_close(first_a, second_a, rtol=0, atol=1e-12)
assert positioned_difference > 1e-12
assert model_input.shape == (BATCH_SIZE, TIME_STEPS, CHANNELS)
print("Check passed: position information distinguishes repeated tokens.")

## What to notice

The token embedding carries symbol identity. The position embedding gives the sequence order information that attention itself does not provide. Their sum is a model-width vector at every batch and time position.

C and Java spell out each table lookup and addition with loops. In PyTorch, `nn.Embedding` performs the indexed lookups and broadcasting lets `[T, C]` position vectors add to `[B, T, C]` token vectors.

References: [PyTorch Embedding](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Embedding.html), [PyTorch arange](https://docs.pytorch.org/docs/2.14/generated/torch.arange.html), and Karpathy's [original notebook](../gpt_dev.ipynb).